<span style = "color: aqua; font-size: 36px;"># DELTA LAKE</span>

Local instance to manage MatrizActividades

## REPORTES EN Excel desde Deltalake

### Deltalake connection

In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
# Importaciones - Ubicacion de DeltaLake - conversion a Pandas
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.styles import Alignment, numbers
from openpyxl.utils import get_column_letter

import shutil
import os
import sys
import time
from pathlib import Path

import pandas as pd
import re
from natsort import order_by_index, index_natsorted
import logging
from deltalake import DeltaTable, write_deltalake
from datetime import datetime
from datetime import date as toDate
import eerssa.utils

logging.basicConfig(level=logging.INFO)

DELTA_TABLE_PATH_ON_HOST = "/home/vlad/delta_V30"

table_path = DELTA_TABLE_PATH_ON_HOST

# ------  Delta Lake  --------- #

if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    df.info()
    print(f"\n\nConectado a la tabla Delta Lake en: {table_path}")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 409648 entries, 0 to 409647
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Item           409648 non-null  int64 
 1   Cuenta         409648 non-null  object
 2   Evento         409648 non-null  object
 3   Actividad      409648 non-null  object
 4   Alimentador    409648 non-null  object
 5   Primario       409648 non-null  object
 6   Desconexion    409648 non-null  object
 7   SIG            409648 non-null  object
 8   Tipo           409648 non-null  object
 9   Materiales     409648 non-null  object
 10  Cuadrilla      409648 non-null  object
 11  Dia            409648 non-null  object
 12  Fecha          409648 non-null  object
 13  InicioEvento   409648 non-null  object
 14  FinEvento      409648 non-null  object
 15  Duracion       409648 non-null  int64 
 16  Responsable    409648 non-null  object
 17  Colaboradores  409648 non-null  int64 
 18  Hora

### Recargar DeltaLake & Dataframe (df)

In [60]:
dt = DeltaTable(table_path)
df = dt.to_pandas()
print(f"VERSION Actual del Deltalake: {dt.version()}")


VERSION Actual del Deltalake: 4788


## EXPORTAR archivo Excel

Creación del archivo Excel donde se guardara el Reporte:
- Es una copia del archivo: `template.xlsx`
- Tiene el formato de la fecha actual

Modificación de la informacion en `df` para acoplarla y sea compatible con Excel


**MEJORAS**
- ~~considerar cambiar el formato de fecha de guion a slash "2026/03/19 11:23:00"~~ 
- ~~verificar que si modifico una fecha en el Excel, esta se importe adecuadamente como String en el Dataframe~~ 
- ~~Duración: transformar de minutos a francción de dia, Dividir para duracion/60*24~~
- ~~Ordenar las ot por items antes de exportar~~
- ~~hay como crear la validacion de datos por listas de una manera programatica?~~
- Cuando exporte la Fecha se escribió bien en Excel, puede ser lo mismo con Duración?
- Verificar en Excel que las Listas sean funcionales, no lo son en LibreOffice (XPS)

Notas:

Cuando modifico una fecha en Excel, se importa en Pandas como `DateTimeObject` por lo que luego lo convierto a `String` previo a volvero a cargar en DeltaLake

<span style="color: red; font-size: 22px;">**DEFINICIÓN DE FECHAS** desde donde y hasta donde se desea generar el reporte</span>


In [5]:
# EJECUTAR: Creación del nuevo archivo desde el template
template_path = 'models/template.xlsx'
he_template_path = 'models/hora_extra_template.xlsx'

today =  datetime.today().strftime('%Y%m%d')
output_filename = f"actividades_{today}.xlsx"
output_he_file  = f"base_HE_{today}.xlsx"

output_path = os.path.join('reporte', output_filename)
output_he   = os.path.join('reporte', output_he_file )

shutil.copyfile(template_path, output_path)
print(f"Se ha copiado Excel para OT en {output_path}")


Se ha copiado Excel para OT en reporte/actividades_20260405.xlsx


In [7]:
reporte_inicia = toDate( 2026, 3, 1)
reporte_finaliza = toDate( 2026,3, 31 )

### Exportar todas las OT entre las fechas

In [ ]:
# Genera un DataFrame entre las fechas para proceder a la exportacion
excel = df.copy()

# ── DataFrame conditioning ──────────────────────────────────────────────────────────
excel['Fecha'] = excel['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
excel['Date']  = excel['Fecha'].apply(lambda x: eerssa.utils.toDateObject( x ))
excel['Duracion'] = excel['Duracion']/(60*24)

filtered_df = excel[ (excel['Date'] >= reporte_inicia ) & ( excel['Date'] <= reporte_finaliza )]
filtered_df = filtered_df.drop(columns=['Date'])
filtered_df.loc[filtered_df['Cuenta'] == 'se_labora', 'HorasExtra'] = 'No'
filtered_df.loc[( filtered_df['Item'] == 1) & (filtered_df['Cuenta'] == 'informativa'), 'HorasExtra'] = 'No'

filtered_df = filtered_df.iloc[index_natsorted(zip(filtered_df['Archivo'], filtered_df['Item']))]

print(f"Dataframe cargado con {len(filtered_df)} filas")

Dataframe cargado con 1845 filas


### Exportar solamente OTs con Horas Extra entre las Fechas

In [63]:
# Genera un DataFrame entre las fechas Y CON HORA EXTRA

excel = df.copy()

# ── DataFrame conditioning ──────────────────────────────────────────────────────────
excel['Fecha'] = excel['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
excel['Date']  = excel['Fecha'].apply(lambda x: eerssa.utils.toDateObject( x ))
excel['Duracion'] = excel['Duracion']/(60*24)

filtered_df = excel[ (excel['Date'] >= reporte_inicia ) & ( excel['Date'] <= reporte_finaliza )]
filtered_df = filtered_df.drop(columns=['Date'])
filtered_df.loc[filtered_df['Cuenta'] == 'se_labora', 'HorasExtra'] = 'No'
filtered_df.loc[( filtered_df['Item'] == 1) & (filtered_df['Cuenta'] == 'informativa'), 'HorasExtra'] = 'No'

# Step 1: Get unique 'id_ot' values where 'HorasExtra' is 'Si'
# We use .loc to filter rows and select the 'id_ot' column, then .unique() to get distinct IDs
ids_with_extra = filtered_df.loc[filtered_df['HorasExtra'] == 'Si', 'id_ot'].unique()

# Step 2: Filter the dataframe to keep all rows where 'id_ot' is in our list
filtered_df = filtered_df[filtered_df['id_ot'].isin(ids_with_extra)]


filtered_df = filtered_df.iloc[index_natsorted(zip(filtered_df['Archivo'], filtered_df['Item']))]

print(f"Dataframe cargado con {len(filtered_df)} filas")

Dataframe cargado con 536 filas


In [ ]:
filtered_df

### Generacion del Excel
Se borran las filas de Template y se insertan nuevas filas.


In [64]:
# Genera el archivo de Excel desde el `filtered_df`

# Load the copied workbook
wb = load_workbook(output_path)

# Access the second sheet (0-based index; change if needed)
sheet = wb.worksheets[1]  # Or wb['Sheet2'] if named

# Optional: Clear existing data from row 2 down (preserves headers and formats)
for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column):
    for cell in row:
        cell.value = None

# Insert DataFrame starting from row 2 (skip headers in DF)
for r_idx, row in enumerate(dataframe_to_rows(filtered_df, index=False, header=False), 2):
    for c_idx, value in enumerate(row, 1):
        sheet.cell(row=r_idx, column=c_idx, value=value)

# Load validation rules
max_row = sheet.max_row

validate_list = '=LISTAS!B$3:B$30' # Cuenta
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'B2:B{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '=LISTAS!E$3:E$30' #Tipo
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'I2:I{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '=LISTAS!G$3:G$30' # Actividad
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'D2:D{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '=LISTAS!I$3:I$100' # Alimentador
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'E2:E{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '"Si,No"' # Binario 
dataValidation = DataValidation( type="list", 
                                formula1=validate_list, 
                                allow_blank=False, 
                                showErrorMessage=True )
dataValidation.add( f'F2:F{max_row}' ) # Primario
dataValidation.add( f'G2:G{max_row}' ) # Desconexion
dataValidation.add( f'H2:H{max_row}' ) # SIG
dataValidation.add( f'S2:S{max_row}' ) # HorasExtra
sheet.add_data_validation(dataValidation)


# Save the modified workbook
wb.save(output_path)
print(f"✅ Se ha generado el archivo de EXCEL en -> {output_path}")


/home/vlad/GIT/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


✅ Se ha generado el archivo de EXCEL en -> reporte/actividades_20260402.xlsx


### Recuperación de Excel

Estas operaciones deben ser revertidas y el volver a calcular la duración del tiempo en minutos

```python 
# Operaciones en version: 0.3.0 <=
excel['Fecha'] = excel['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
excel['Duracion'] = excel['Duracion']/(60*24)  # Volver a calcular


```

<span style="color: red; font-size: 22px;">**NOMBRE DEL ARCHIVO** para cargar de Excel a Pandas</span>

In [65]:
#excel_file_path = 'reporte/20260319_reporte_he_marzo15.xlsx'

excel_file_path = output_path


In [66]:
# 1. Load the modified Excel file
modified_df = pd.read_excel(excel_file_path, sheet_name = "ACTIVIDADES")

# 2. Se vuelva a colocar el String de TimeZone en la Fecha
modified_df['Fecha'] = modified_df['Fecha'].apply(lambda x: eerssa.utils.ColocarTimezone( x ))

# 3. Convertir de String a TimeObject y se vuelve a calcular la duración en minutos
modified_df['Ini'] = pd.to_datetime(modified_df['InicioEvento'], errors='coerce')
modified_df['Fin'] = pd.to_datetime(modified_df['FinEvento'], errors='coerce')

modified_df['Duracion'] = eerssa.utils.calcular_minutos_transcurridos(
    modified_df['Ini'],
    modified_df['Fin']
)

modified_df = modified_df.drop(columns=['Ini', 'Fin'])

#4. Convert everything to datetime objects first, then format them all as uniform strings
modified_df['InicioEvento'] = pd.to_datetime(modified_df['InicioEvento'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
modified_df['FinEvento'] = pd.to_datetime(modified_df['FinEvento'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')


# 4. Ensure Schema Consistency
# Excel often introduces new columns (like empty comments) or reorders them.
# We force the modified_df to have the same columns as the original df.
modified_df = modified_df[df.columns]


### Actulizar en DeltaLake

In [73]:
# Con el archivo modificado en Excel, se actualizan las filas en DeltaLake
try:
    # --- Start of new logic ---
    # 1. Get a list of all unique 'id_ot' values from the source DataFrame.
    ids_to_update = modified_df['id_ot'].unique()

    # 2. Format the list into a SQL-compatible string like "(101, 102, 103)".
    # This is crucial for the IN clause to work correctly.
    ids_predicate_string = ", ".join(map(str, ids_to_update))
    
    # 3. Define the delete predicate to scope deletions to only the OTs being updated.
    delete_predicate = f"target.id_ot IN ({ids_predicate_string})"
    # --- End of new logic ---

    # The unique key for matching rows remains the same.
    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
            source=modified_df,
            predicate=unique_key_predicate,
            source_alias="source",
            target_alias="target"
        )
        .when_matched_update_all()  # Rule 1: If a row exists, update it.
        .when_not_matched_insert_all()  # Rule 2: If it's a new row, insert it.
        .when_not_matched_by_source_delete(  # Rule 3: If an old row is now gone...
            predicate=delete_predicate  # ...delete it, but ONLY if it belongs to an OT we are modifying.
        )
        .execute()
    )
    saved = "✅ **Successfully saved changes for all modified OTs to Delta Lake!**"
except Exception as e:
    saved = f"❌ **Error saving to Delta Lake:** {e}"

print(saved)

dt = DeltaTable(table_path)
df = dt.to_pandas()
print(f"VERSION Actual del Deltalake: {dt.version()}")


✅ **Successfully saved changes for all modified OTs to Delta Lake!**
VERSION Actual del Deltalake: 4791


### Generar Reporte_HE

In [74]:
# Creación de Dataframe he
# Genera un DataFrame entre las fechas para proceder a la exportacion
he = df.copy()

# ── DataFrame conditioning ──────────────────────────────────────────────────────────
he['Fecha'] = he['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
he['Date']  = he['Fecha'].apply(lambda x: eerssa.utils.toDateObject( x ))
he['InicioEvento'] = he['InicioEvento'].apply( lambda x: eerssa.utils.elimina_timezone( x ) )
he['FinEvento'] = he['FinEvento'].apply( lambda x: eerssa.utils.elimina_timezone( x ) )
he['Duracion'] = he['Duracion']/(60*24)

he = he[ 
    (he['Date'] >= reporte_inicia   ) &   # Fecha Inicial
    (he['Date'] <= reporte_finaliza ) &   # Fecha Final
    (he['HorasExtra']== 'Si' )            # Es Hora Extra
    ]

he = he[[ 'Cuadrilla', 'Responsable', 'Dia', 'Date','Item', 'InicioEvento', 'FinEvento', 'Duracion', 'Evento','Cuenta','id_ot','Archivo' ]]


In [75]:
# Agrupar Eventos. OPPUS
# Parse datetime columns
he['InicioEvento'] = pd.to_datetime(he['InicioEvento'])
he['FinEvento'] = pd.to_datetime(he['FinEvento'])

# Truncate to the minute
he['Inicio_min'] = he['InicioEvento'].dt.floor('min')
he['Fin_min'] = he['FinEvento'].dt.floor('min')

# Sort
he = he.sort_values(['id_ot', 'InicioEvento']).reset_index(drop=True)

# Detect breaks: new group starts when id_ot changes OR there's a time gap
new_group = (
    (he['id_ot'] != he['id_ot'].shift()) |
    (he['Inicio_min'] != he['Fin_min'].shift())
)

# Assign group number using cumulative sum of breaks
he['group'] = new_group.cumsum()

# Now group
result = he.groupby(['group']).agg(
    Cuadrilla    = ('Cuadrilla', 'first'),
    Responsable  = ('Responsable', 'first'),
    Dia          = ('Dia','first'),
    Date         = ('Date','first'),
    InicioEvento = ('InicioEvento', 'min'),    # Start of the group
    FinEvento    = ('FinEvento', 'max'),       # End of the group
    Duracion     = ('Duracion', 'sum'),        # Total duration
    Evento       = ('Evento', list),           # All events in the group
    Cuenta       = ('Cuenta', list),
    id_ot        = ('id_ot','first'),
    Items        = ('Item',list),
    Num_Filas    = ('Evento', 'count'),         # How many rows in the group
    Archivo      = ('Archivo','first')
).reset_index().drop(columns='group').query('Duracion != 0.0').sort_values(['Archivo', 'InicioEvento']).reset_index(drop=True)

# Limpiar el texto y limpiar las cuentas
result['Evento'] = result['Evento'].apply(lambda x: eerssa.utils.limpiar_lista_eventos(x))
result['Cuenta'] = result['Cuenta'].apply(lambda x: eerssa.utils.limpiar_cuentas(x))
result['Items']  = result['Items'].apply(lambda x: eerssa.utils.limpiar_items(x))
result[['InicioEvento', 'FinEvento']] = result[['InicioEvento', 'FinEvento']].apply(lambda x: x.dt.time) #TimeObjects

# Create the formula string for every row, starting at index 2
result['Duracion'] = [f'=F{i}-E{i}' for i in range(2, len(result) + 2)]


In [76]:
result

,Cuadrilla,Responsable,Dia,Date,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo
0,Jefatura Zonal Zamora,PALACIOS MERINO ERNESTO VLADIMIR,lunes,2026-03-30,06:30:00,08:00:00,=F2-E2,"Desde Zamora hasta Loja, talleres de la EERSSA",?,173821,1,1,OT [00] 00_Jefe Zonal 2026-03-30 (004) EP.pdf
1,Zamora Z1 (Cuadrilla. Nro. 6),"RS, VZH",lunes,2026-03-23,17:05:00,21:47:00,=F3-E3,DAÑOS REPORTADOS POR Centro de Control Nos tra...,"ACOMETIDAS, REDES",173271,"14, 15, 16, 17, 18",5,OT [01] Cuadrilla Zamora 2026-03-23 (006) FR.pdf
2,Zamora Z1 (Cuadrilla. Nro. 6),"FR, RS, LL, VZH, JCR",martes,2026-03-24,16:00:00,22:41:00,=F4-E4,Desde Nambija nos trasladamos hacia Namirez Al...,"REDES, MEDIDORES",173348,"8, 9, 10, 13, 14, 15, 16, 17, 18, 19",10,OT [01] Cuadrilla Zamora 2026-03-24 (006) FR.pdf
3,Zamora Z1 (Cuadrilla. Nro. 6),"RS, VZH",miércoles,2026-03-25,17:00:00,19:27:00,=F5-E5,DAÑOS REPORTADOS POR Centro de Control Nos tra...,?,173441,"14, 15, 16",3,OT [01] Cuadrilla Zamora 2026-03-25 (006) FR.pdf
4,Zamora Z1 (Cuadrilla. Nro. 6),"RS, VZH",jueves,2026-03-26,20:55:00,23:33:00,=F6-E6,DAÑO REPORTADO POR Centro de Control Nos trasl...,REDES,173504,"16, 17, 18",3,OT [01] Cuadrilla Zamora 2026-03-26 (006) FR.pdf
5,Zamora Z1 (Cuadrilla. AP Nro. 4),"LM, MC",martes,2026-03-24,19:01:00,22:05:00,=F7-E7,Se hase recorrido nocturno levantando informac...,ALUMBRADO,173347,28,1,OT [02] Alumbrado Zamora 2026-03-24 (012) LM.pdf
6,Zamora Z1 (Cuadrilla. AP Nro. 4),"LM, MC",jueves,2026-03-26,18:48:00,22:03:00,=F8-E8,"Zamora, trabajo programado y autorizado por su...",?,173497,17,1,OT [02] Alumbrado Zamora 2026-03-26 (012) LM.pdf
7,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, WC, NR",lunes,2026-03-23,17:00:00,20:18:00,=F9-E9,DAÑO REPORTADO POR Centro de Control. Mensaje ...,"ACOMETIDAS, MEDIDORES",173374,"8, 9, 11, 12, 14",5,OT [03] Cuadrilla Yacuambi 2026-03-23 (017) NL...
8,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, WC, NR",miércoles,2026-03-25,15:00:00,18:08:00,=F10-E10,Desde Guaguayme Alto nos trasladamos a la Agen...,?,173550,11,1,OT [03] Cuadrilla Yacuambi 2026-03-25 (017) NL...
9,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, WC, VZH, NR",jueves,2026-03-26,16:30:00,18:37:00,=F11-E11,Desde Guagauyme Alto se retorna a Yacuambi,?,173588,9,1,OT [03] Cuadrilla Yacuambi 2026-03-26 (017) NL...


#### Generar Excel Base de Horas Extra

In [77]:
# Genera el archivo de Excel desde el `result`

# Copia el Template
shutil.copyfile(he_template_path, output_he)
print(f"Se ha copiado Excel para HE en {output_he}")

# Load the copied workbook
wb = load_workbook(output_he)

# Access the second sheet (0-based index; change if needed)
sheet = wb.worksheets[1]  # Or wb['result'] if named

# Optional: Clear existing data from row 2 down (preserves headers and formats)
for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column):
    for cell in row:
        cell.value = None

# Insert DataFrame starting from row 2 (skip headers in DF)
for r_idx, row in enumerate(dataframe_to_rows(result, index=False, header=False), 2):
    for c_idx, value in enumerate(row, 1):
        sheet.cell(row=r_idx, column=c_idx, value=value)

# Save the modified workbook
wb.save(output_he)
print(f"✅ Se ha generado el archivo de EXCEL en -> {output_he}")


Se ha copiado Excel para HE en reporte/base_HE_20260402.xlsx
✅ Se ha generado el archivo de EXCEL en -> reporte/base_HE_20260402.xlsx


/home/vlad/GIT/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


<span style="color: red; font-size: 22px;">**NOMBRE DEL ARCHIVO** para cargar de Base Horas Extra Excel a Pandas</span>

In [8]:
# ESTE ARCHIVO DEBERÁ SER CAMBIADO, En testing usamos el template (marzo)
archivo_base_he = os.path.join('models','hora_extra_template.xlsx')

In [13]:
festivos = pd.read_excel(
    archivo_base_he,
    sheet_name='Revisar_primero',
    usecols='A:C',      # Only columns A and B
    header=0            # First row as column names
)

festivos

,Aplica,Fecha,Etiqueta
0,TODOS,2026-01-01,“Día Festivo: Año Nuevo”
1,TODOS,2026-02-13,“Día Festivo: Día del Oriente Ecuatoriano”
2,TODOS,2026-02-16,“Día Festivo: Carnaval (Lunes)”
3,TODOS,2026-02-17,“Día Festivo: Carnaval (Martes)”
4,TODOS,2026-04-03,“Día Festivo: Viernes Santo”
5,TODOS,2026-05-01,“Día Festivo: Día del Trabajo”
6,TODOS,2026-05-25,“Día Festivo: Batalla de Pichincha”
7,TODOS,2026-08-10,“Día Festivo. Primer Grito de Independencia”
8,TODOS,2026-10-09,“Día Festivo: Independencia de Guayaquil”
9,TODOS,2026-11-02,“Día Festivo: Santos Difuntos”


In [ ]:
noche = pd.read_excel(
    archivo_base_he,
    sheet_name='Revisar_primero',
    usecols='D',      # Only columns A and B
    header=0            # First row as column names
).squeeze("columns") # Turns the 1-column DataFrame into a Series

noche

0    2026-01-12
1    2026-01-13
2    2026-01-14
3    2026-01-15
4    2026-01-16
5    2026-02-02
6    2026-02-03
7    2026-02-04
8    2026-02-05
9    2026-02-23
10          NaT
11          NaT
12          NaT
13          NaT
14          NaT
15          NaT
16          NaT
17          NaT
18          NaT
19          NaT
20          NaT
21          NaT
22          NaT
23          NaT
24          NaT
25          NaT
26          NaT
Name: TurnoAP, dtype: datetime64[ns]

In [ ]:
# 1. Load the workbook manually to access formulas
# data_only=False is CRITICAL here to get the formula string instead of the result
wb = load_workbook(archivo_base_he, data_only=False)

festivos = pd.read_excel(
    archivo_base_he,
    sheet_name='Revisar_primero',
    usecols='A:C',      # columns A to C
    header=0            # First row as column names
)

noche = pd.read_excel(
    archivo_base_he,
    sheet_name='Revisar_primero',
    usecols='D',        # Only column D - days with change of labor
    header=0            # First row as column names
).squeeze("columns") # Turns the 1-column DataFrame into a Series


# 2. Select the "result" sheet
if "result" in wb.sheetnames:
    ws = wb["result"]
else:
    ws = wb.active   # Lanzar Excepcion


# 3. Convert the worksheet data into a Pandas DataFrame
data = ws.values
cols = next(data)  # Extract the first row as header
base = pd.DataFrame(data, columns=cols)
base = base.dropna(subset=['Cuadrilla'])

# 2. Convert Column G (Duracion) back to formula strings 
# (If read_excel evaluates it, this ensures we treat it as text/object)
base['Duracion'] = base['Duracion'].astype(str)

# 3. Convert Column from "string with commas" back to Python Lists
list_cols = ['Responsable','Items']
for col in list_cols:
    # We split by ", " and handle potential NaN values
    base[col] = base[col].apply(lambda x: str(x).split(", ") if pd.notnull(x) else [])

# Diccionarios para las cuentas. 
base['Cuenta'] = base['Cuenta'].apply(eerssa.utils.cuenta_to_dict)

# 4. Handle Column D (Date)
base['Fecha'] = pd.to_datetime(base['Fecha']).dt.date

# 5. Handle Columns E, F (Time)
# We convert them to strings or stay as datetime.time
base['InicioEvento'] = pd.to_datetime(base['InicioEvento'], format='%H:%M:%S').dt.time
base['FinEvento'] = pd.to_datetime(base['FinEvento'], format='%H:%M:%S').dt.time


# --- CALCULAR EL TIPO DE REMUNERACION ---
# Ensure date columns are proper datetime types
festivos['Fecha'] = pd.to_datetime(festivos['Fecha']).dt.date

# --- Build lookup sets ---
# Condition 1: Dates that apply to EVERYONE
festivos_todos = set(festivos.loc[festivos['Aplica'] == 'TODOS', 'Fecha'])

# Condition 2: Dates specific to a Cuadrilla (CANTONIZACION)
# Build a set of (Cuadrilla, Fecha) tuples for quick lookup
festivos_especificos = set(
    zip(
        festivos.loc[festivos['Aplica'] != 'TODOS', 'Aplica'],
        festivos.loc[festivos['Aplica'] != 'TODOS', 'Fecha']
    )
)

# --- Classification function ---
# TODO: falta los dias de Turno de Alumbrado. 

def clasificar_tipo(row):
    fecha     = row['Fecha']
    cuadrilla = row['Cuadrilla']
    dia       = row['Dia']
    inicio    = row['InicioEvento']

    # 1. FESTIVO — national/universal rest day
    if fecha in festivos_todos:
        return 'FESTIVO'

    # 2. CANTONIZACION — rest day for this specific Cuadrilla
    if (cuadrilla, fecha) in festivos_especificos:
        return 'CANTONIZACION'

    # 3. DESCANSO — weekend
    if dia in ('sábado', 'domingo'):
        return 'DESCANSO'

    # 4. MAD — started before 6am
    if inicio.hour < 6:
        return 'MAD'

    # 5. Default
    return 'NORMAL'

# --- Apply ---
base['Tipo'] = base.apply(clasificar_tipo, axis=1)


/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [14]:
base.sample(10)

,Cuadrilla,Responsable,Dia,Fecha,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
187,Yantzaza Z1 (Cuadrilla. Nro. 5),"[AO, JLC]",jueves,2026-03-26,17:00:00,19:26:00,=F189-E189,"El Pindal, se revisa LMV desde la estructura 3...","[{'cuenta': 'REDES', 'peso': 1.0}]",173753,"[1, 3, 5]",3,OT [04] Cuadrilla Yantzaza 2026-03-26 (023) AO...,NORMAL
121,El Pangui Z1 (Cuadrilla. Nro. 4),"[HM, AD]",lunes,2026-03-16,19:28:00,19:58:00,=F123-E123,"El Pangui, la estructura 227941 se rehabilita ...","[{'cuenta': 'REDES', 'peso': 1.0}]",172771,[],3,OT [08] Cuadrilla El Pangui 2026-03-16 (042) H...,NORMAL
8,Zamora Z1 (Cuadrilla. Nro. 6),"[RS, JCR]",domingo,2026-03-08,23:55:00,23:59:00,=F10-E10,"Tranposrte, nos trasladamos desde agencia EERS...","[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",172769,[],1,OT [01] Cuadrilla Zamora 2026-03-08 (007) RS.pdf,DESCANSO
13,Zamora Z1 (Cuadrilla. Nro. 6),"[LL, SO]",lunes,2026-03-16,17:05:00,18:39:00,=F15-E15,"Sakantza, M# 18-237293 breaker dañado, se camb...","[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",172774,[],1,OT [01] Cuadrilla Zamora 2026-03-16 (006) FR.pdf,NORMAL
31,Zamora Z1 (Cuadrilla. AP Nro. 4),"[LM, MC]",jueves,2026-03-19,17:00:00,18:30:00,=F33-E33,"INC No. 11041012368. Atendido, luminaria de 70...","[{'cuenta': 'ALUMBRADO', 'peso': 1.0}]",172959,[],2,OT [02] Alumbrado Zamora 2026-03-19 (012) LM.pdf,NORMAL
0,Zamora Z1 (Cuadrilla. Nro. 6),"[RS, JCR]",lunes,2026-03-02,21:27:00,23:59:00,=F2-E2,"San Carlos de las Minas, arreglo de conexiones...","[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",172309,[],3,OT [01] Cuadrilla Zamora 2026-03-02 (007) RS.pdf,NORMAL
59,Yantzaza Z1 (Cuadrilla. Nro. 5),"[SB, GT]",martes,2026-03-17,19:16:00,21:45:00,=F61-E61,"Guayacanes, en medidor 1262778, se revisa volt...","[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",172872,[],5,OT [04] Cuadrilla Yantzaza 2026-03-17 (021) SB...,NORMAL
124,El Pangui Z1 (Cuadrilla. Nro. 4),"[HM, AD]",viernes,2026-03-20,06:52:00,08:00:00,=F126-E126,"Atención de daños en Tsarunts, desde El Pangui...","[{'cuenta': 'REDES', 'peso': 1.0}]",173109,[],1,OT [08] Cuadrilla El Pangui 2026-03-20 (042) H...,NORMAL
72,Líneas Energizadas (Cuadrilla Nro.6),[GLE],martes,2026-03-03,06:40:00,08:00:00,=F74-E74,San Roque: estructura # 311147 se puentea deri...,"[{'cuenta': 'REDES', 'peso': 1.0}]",171894,[],1,OT [05] Energizados Yantzaza 2026-03-03 (027) ...,NORMAL
194,Líneas Energizadas (Cuadrilla Nro.6),[GLE],lunes,2026-03-23,17:00:00,18:00:00,=F196-E196,Es s/n se instala 3 estribos y conexión de 3 p...,"[{'cuenta': 'REDES', 'peso': 1.0}]",173230,[4],1,OT [05] Energizados Yantzaza 2026-03-23 (027) ...,NORMAL


In [52]:
# Dividir para cada persona

# 1. Extract all unique labels (initials) from base lists
unique_responsables = base['Responsable'].explode().dropna().unique()

# 2. Initialize your list of DataFrames
horasExtra = []

# 3. Initialize a dictionary to map names to DataFrames
horasExtra_todos = {}

# 4. Loop through each unique person
for person in unique_responsables:

    # 5. Filter base DataFrame to keep only rows where the person is in the list
    filtered_df = base[base['Responsable'].apply(lambda x: person in x if isinstance(x, list) else False)].copy()

    # 6. Reset index so rows are numbered from 0
    filtered_df = filtered_df.reset_index(drop=True)

    # 7. Recalculate Duracion as Excel formulas based on NEW row positions
    #    +2 accounts for: 1 header row + Excel's 1-based index
    e_col = 'E'  # InicioEvento column letter in Excel
    f_col = 'F'  # FinEvento column letter in Excel

    filtered_df['Duracion'] = [
        f'={f_col}{i + 2}-{e_col}{i + 2}'
        for i in range(len(filtered_df))
    ]

    # 8. Store the filtered DataFrame
    horasExtra.append(filtered_df)
    horasExtra_todos[person] = filtered_df

In [55]:
horasExtra_todos['FG']

,Cuadrilla,Responsable,Dia,Fecha,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, CLP, MA]",lunes,2026-03-02,17:00:00,18:05:00,=F2-E2,"San Francisco, se revisa red de MV y se encuen...",[REDES],171814.0,None,2.0,OT [09] Cuadrilla Gualaquiza 2026-03-02 (045) ...,NORMAL
1,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, CLP, MA]",martes,2026-03-03,17:00:00,19:34:00,=F3-E3,"Sector Zapotillo, se desbroza vegetación, se r...",[REDES],171926.0,None,3.0,OT [09] Cuadrilla Gualaquiza 2026-03-03 (045) ...,NORMAL
2,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, CLP]",miércoles,2026-03-04,19:31:00,20:51:00,=F4-E4,"Sector Pasaje, se revisa medidor 1000422986 es...",[REDES],172010.0,None,3.0,OT [09] Cuadrilla Gualaquiza 2026-03-04 (045) ...,NORMAL
3,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, CLP, MA]",jueves,2026-03-05,17:00:00,18:03:00,=F5-E5,"Las Peñas, en la estructura 64727 se encuentra...",[REDES],172104.0,None,2.0,OT [09] Cuadrilla Gualaquiza 2026-03-05 (045) ...,NORMAL
4,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, MA]",viernes,2026-03-06,19:19:00,20:39:00,=F6-E6,"El Belén, en la estructura 127785 se revisa RB...",[REDES],172194.0,None,3.0,OT [09] Cuadrilla Gualaquiza 2026-03-06 (045) ...,NORMAL
5,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, MA]",sábado,2026-03-07,11:22:00,15:21:00,=F7-E7,"El Belén, domicilio de Gózalo Orellana, medido...",[ACOMETIDAS],172256.0,None,6.0,OT [09] Cuadrilla Gualaquiza 2026-03-07 (045) ...,DESCANSO
6,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, MA]",domingo,2026-03-08,17:26:00,19:36:00,=F8-E8,"La Pradera, en la estructura 265102 se revisa ...",[REDES],172301.0,None,3.0,OT [09] Cuadrilla Gualaquiza 2026-03-08 (045) ...,DESCANSO
7,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, CLP, MA]",jueves,2026-03-12,17:15:00,19:26:00,=F9-E9,"El Rosario, se realiza montaje del transformad...",[REDES],172569.0,None,3.0,OT [09] Cuadrilla Gualaquiza 2026-03-12 (045) ...,NORMAL
8,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, MA, CLP]",miércoles,2026-03-18,07:18:00,08:00:00,=F10-E10,"Tutus, se revisa red de MV se encuentra quemad...",[REDES],172943.0,None,2.0,OT [09] Cuadrilla Gualaquiza 2026-03-18 (045) ...,NORMAL
9,Gualaquiza Z1 (Cuadrilla. Nro. 3),"[FG, CLP, MA]",viernes,2026-03-20,07:00:00,08:00:00,=F11-E11,"Tsaurunts, en coordinacion con Henry Mendieta ...",[REDES],173104.0,None,3.0,OT [09] Cuadrilla Gualaquiza 2026-03-20 (045) ...,NORMAL


#### GENERAR Horas Extra individual

<span style="color: red; font-size: 22px;">**NOMBRE DEL ARCHIVO** para guardar de Base Horas Extra Excel a Pandas</span>

In [43]:
guardar_he_todos = os.path.join('reporte', 'Reporte_he_todos.xlsx')


In [56]:
# Generar archivo de todos con Horas Extra

# Use Pandas ExcelWriter with the openpyxl engine
with pd.ExcelWriter(guardar_he_todos, engine='openpyxl') as writer:
    
    for person, df in horasExtra_todos.items():
        
        # 1. Sort the DataFrame by Date
        df_sorted = df.sort_values(by='Fecha')
        
        # 2. Write to Excel
        df_sorted.to_excel(writer, sheet_name=person, index=False)
        
        # 3. Access the underlying openpyxl worksheet
        worksheet = writer.sheets[person]

        # 4. Get column indexes (1-based) for Evento and Duracion
        evento_col_idx   = df_sorted.columns.get_loc('Evento') + 1
        duracion_col_idx = df_sorted.columns.get_loc('Duracion') + 1
        evento_col_letter   = get_column_letter(evento_col_idx)
        duracion_col_letter = get_column_letter(duracion_col_idx)

        # 5. Auto-fit all columns based on content, except Evento (fixed 14cm ≈ 53 units)
        for col in worksheet.columns:
            col_letter = get_column_letter(col[0].column)
            if col_letter == evento_col_letter:
                worksheet.column_dimensions[col_letter].width = 53  # ~14cm
            else:
                max_length = max(
                    (len(str(cell.value)) if cell.value is not None else 0)
                    for cell in col
                )
                worksheet.column_dimensions[col_letter].width = min(max_length + 2, 60)

        # 6. Format Duracion column (G) as time [h]:mm:ss
        time_format = '[h]:mm'
        for row in range(2, len(df_sorted) + 2):  # Skip header row
            cell = worksheet.cell(row=row, column=duracion_col_idx)
            cell.number_format = time_format

        # 7. Fecha merging logic (your existing code, unchanged)
        fecha_col_idx = df_sorted.columns.get_loc('Fecha') + 1
        start_row = 2
        max_row = len(df_sorted) + 1
        current_date = worksheet.cell(row=start_row, column=fecha_col_idx).value

        for row in range(3, max_row + 2):
            cell_value = worksheet.cell(row=row, column=fecha_col_idx).value if row <= max_row else None

            if cell_value != current_date:
                if row - start_row > 1:
                    worksheet.merge_cells(
                        start_row=start_row,
                        start_column=fecha_col_idx,
                        end_row=row - 1,
                        end_column=fecha_col_idx
                    )
                    worksheet.cell(row=start_row, column=fecha_col_idx).alignment = Alignment(
                        horizontal='center', vertical='center'
                    )
                current_date = cell_value
                start_row = row

-----------------

<span style="color: BLUE; font-size: 26px;">**FIN DE REPORTES EN EXCEL** </span>


-----------------------------

# Full Mogno+Delta

In [22]:
import sys
import time
from pathlib import Path
import pandas as pd
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)

DELTA_TABLE_PATH_ON_HOST = "/home/vlad/delta_V30"

table_path = DELTA_TABLE_PATH_ON_HOST


Success!!!


### Conexon con MongoDB

In [23]:
# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


### Conexion con DeltaLake

In [24]:
# DELTA LAKE Connection

# Verify the existence of the DELTA LAKE table
if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")



Conectado a la tabla Delta Lake en: /home/vlad/delta_V30


In [25]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 411127 entries, 0 to 411126
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Item           411127 non-null  int64 
 1   Cuenta         411127 non-null  object
 2   Evento         411127 non-null  object
 3   Actividad      411127 non-null  object
 4   Alimentador    411127 non-null  object
 5   Primario       411127 non-null  object
 6   Desconexion    411127 non-null  object
 7   SIG            411127 non-null  object
 8   Tipo           411127 non-null  object
 9   Materiales     411127 non-null  object
 10  Cuadrilla      411127 non-null  object
 11  Dia            411127 non-null  object
 12  Fecha          411127 non-null  object
 13  InicioEvento   411127 non-null  object
 14  FinEvento      411127 non-null  object
 15  Duracion       411127 non-null  int64 
 16  Responsable    411127 non-null  object
 17  Colaboradores  411127 non-null  int64 
 18  Hora

In [26]:
df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,5,MEDIDORES,Chicaña se revisa medidor 19-207825 se encuen...,NO PROG,Los Encuentros,No,No,No,PREVENTIVO,·,...,2024-11-03 10:15:00,2024-11-03 10:55:00,40,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,4-100,YANTZAZA - MUTINZA - CHICAÑA - LA CETZA - MU...,140743,OT [04] Cuadrilla Yantzaza 2024-11-03 (032) AO...
1,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,OT [21] Agencia Zamora 2022-02-11 (027) RM.pdf
2,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,10_Test Orden de trabajo Zamora 11-02-2022 (RM...
3,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,OT [21] Agencia Zamora 2022-02-11 (027) RM.pdf
4,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,10_Test Orden de trabajo Zamora 11-02-2022 (RM...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411122,1,informativa,DAÑO REPORTADO POR EL CENTRO DE CONTROL,INFO,·,No,No,No,·,·,...,2026-02-09 00:00:01,2026-02-09 00:00:02,0,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf
411123,2,transporte,Nos trasladamos desde la agencia EERSSA Zamora...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-09 19:42:00,2026-02-09 20:20:00,38,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf
411124,3,REDES,"Guadalupe, La Libertad en la estructura. 25893...",NO PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2026-02-09 20:20:00,2026-02-09 21:45:00,85,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf
411125,4,transporte,Nos trasladamos desde La Libertad a la agencia...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-09 21:45:00,2026-02-09 22:19:00,34,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf


### Analisis de Cambios en una OT

In [25]:
#  ANALIZAR UNA OT filtrando por su indice

filtered_df = df.query( f"id_ot == 159262" ).sort_values(by='Item')
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
406752,1,informativa,Se coordina los trabajos del dia con la admini...,PROG,·,No,No,No,RUTINARIA,·,...,2025-08-15 08:00:00,2025-08-15 08:30:00,30,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
269326,2,ACOMETIDAS,En La Palmira se realiza inspección para un N/...,PROG,Bomboiza,No,No,No,EXPANSION,·,...,2025-08-15 08:30:00,2025-08-15 09:00:00,30,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
285662,3,ACOMETIDAS,En Los Bayanes se realiza inspección para un N...,PROG,Bomboiza,No,No,No,EXPANSION,·,...,2025-08-15 09:00:00,2025-08-15 09:30:00,30,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
252986,4,ACOMETIDAS,En Abig Emanuel se realiza inspección para un ...,PROG,Bomboiza,No,No,No,EXPANSION,·,...,2025-08-15 09:30:00,2025-08-15 10:00:00,30,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
236650,5,transporte,Traslado de Abig Emanuel a Chanzas,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-08-15 10:00:00,2025-08-15 11:30:00,90,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
301993,6,REDES,DAÑO REPORTADO POR CCEn Chanzas se encuentra c...,NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2025-08-15 11:30:00,2025-08-15 16:00:00,270,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
0,9,lunch,LUNCH EN CHANZAS NO SE REGISTRO EN EL RELOJ BI...,ALIMEN,·,No,No,No,LUNCH,·,...,2025-08-15 16:00:00,2025-08-15 17:00:00,60,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
386174,10,transporte,Traslado de Chanzas a el Pangui,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-08-15 17:00:00,2025-08-15 18:28:00,88,AMBULUDI SILVA FAUSTO JOEL,1,Si,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf
220314,12,se_labora,Se labora F.A y L.V en horario de 08:00 a 16:0...,LABORA,·,No,No,No,·,·,...,2025-08-15 00:00:01,2025-08-15 00:00:02,0,AMBULUDI SILVA FAUSTO JOEL,1,Si,4-118,El Pangui - Chanzas - Los Bayanes - Abig Emanuel,159262,OT [23] Agencia El Pangui 2025-08-15 (058) FA.pdf


In [26]:
#filtered_df = df.query( f"Evento == Se labora F.A y L.V en horario de 08:00 a 13:00 y de 14:00 a 18:38" ).sort_values(by='Item')
filtered_df = df[df["Evento"].str.startswith("Se labora F.A y L.V en horario de 08:00 a 13:00 y de 14:00 a 18:38")].sort_values(by='Item')
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
169113,16,se_labora,Se labora F.A y L.V en horario de 08:00 a 13:0...,LABORA,·,No,No,No,·,·,...,2025-08-14 00:00:01,2025-08-14 00:00:02,0,AMBULUDI SILVA FAUSTO JOEL,1,Si,4-118,El Pangui - La Alfonsina - Los Bayanes - Abig ...,159195,OT [23] Agencia El Pangui 2025-08-14 (055) FA.pdf


### Volver a carga DeltaLake

In [28]:
dt = DeltaTable(table_path)
df = dt.to_pandas()
dt.version()


4185

#### Historia de Delta Lake

In [29]:
print("\n--- Table History ---")
history = dt.history(limit = 10)
for commit in history:
    print(
        f"Version: {commit['version']}, "
        f"Timestamp: {datetime.fromtimestamp(commit['timestamp']/1000)}, "
        f"Operation: {commit['operation']}"
    )



--- Table History ---
Version: 4185, Timestamp: 2026-02-24 15:14:16.833000, Operation: MERGE
Version: 4184, Timestamp: 2026-02-24 14:25:26.645000, Operation: MERGE
Version: 4183, Timestamp: 2026-02-24 14:23:47.639000, Operation: MERGE
Version: 4182, Timestamp: 2026-02-24 14:23:10.838000, Operation: WRITE
Version: 4181, Timestamp: 2026-02-24 10:49:05.109000, Operation: WRITE
Version: 4180, Timestamp: 2026-02-24 08:12:56.622000, Operation: MERGE
Version: 4179, Timestamp: 2026-02-24 08:11:35.811000, Operation: WRITE
Version: 4178, Timestamp: 2026-02-24 08:10:34.795000, Operation: WRITE
Version: 4177, Timestamp: 2026-02-24 08:09:48.785000, Operation: WRITE
Version: 4176, Timestamp: 2026-02-24 08:09:11.792000, Operation: WRITE


### Retaurar a una version anterior de Deltalake - Timetravel

In [ ]:
target_version = 2620

# --- Restore using a version number ---
dt.restore(target_version)
        
print(f"✅ Restauración exitosa! La tabla se encuentra en la version {dt.version()}.")

### Optimización y Aspirado

In [ ]:
dt.optimize.compact()

{'numFilesAdded': 1,
 'numFilesRemoved': 402,
 'filesAdded': '{"avg":15978605.0,"max":15978605,"min":15978605,"totalFiles":1,"totalSize":15978605}',
 'filesRemoved': '{"avg":76402.73631840796,"max":8811865,"min":7966,"totalFiles":402,"totalSize":30713900}',
 'partitionsOptimized': 1,
 'numBatches': 779,
 'totalConsideredFiles': 402,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': True}

In [ ]:
dt.vacuum(retention_hours=1080, enforce_retention_duration=False, dry_run=True)


[2025-09-29T19:30:20Z WARN  deltalake_core::kernel::transaction] Attempting to write a transaction 2464 but the underlying table has been updated to 2464
    DefaultLogStore(/home/vlad/delta_V30/)


['part-00001-d57c30d9-10a3-494d-a0a7-16d3282b16af-c000.snappy.parquet',
 'part-00001-a0a15b0d-2812-4135-9fe4-5519df35c7b3-c000.snappy.parquet',
 'part-00001-7dd7a227-d121-493f-b29f-4175e0e384e1-c000.snappy.parquet',
 'part-00001-0f53da6e-5d65-42ce-ac8b-54bc8efc5f5e-c000.snappy.parquet',
 'part-00001-c0d7b295-f7b8-45ce-934c-e78009ca4b45-c000.snappy.parquet',
 'part-00001-7829cf16-b846-4b08-8c3e-6f0a58ea448c-c000.snappy.parquet',
 'part-00001-bf1fb039-dce0-4031-8a66-389a2fcc38ba-c000.snappy.parquet',
 'part-00001-5ee1ed22-402f-4d6a-b449-3ec7e2ec0073-c000.snappy.parquet',
 'part-00001-3e288d50-1c82-463b-89d6-896f026a6ca0-c000.snappy.parquet',
 'part-00001-42699686-c7a4-488e-9690-2e9d5daa5edf-c000.snappy.parquet',
 'part-00001-992c3e68-e768-47a4-8336-b3e1628ea166-c000.snappy.parquet',
 'part-00001-1f140a4a-5108-461c-a755-075b4f84be95-c000.snappy.parquet',
 'part-00001-e6c7f175-e51f-4070-888b-d90b1ad0c15a-c000.snappy.parquet',
 'part-00001-e9af8fb1-8be0-4fd8-9f1f-6aad6e59f61b-c000.snappy.pa

## Limpieza de dataset

In [ ]:
df

### Cambio de Tipo de Cuenta mal escrita

Hay casos en los que las Cuentas se guardan con espacios al final. por ejemplo `'CORRECTIVO '` y también `'CORRECTIAS '`
Se ha corregido este error a partir de la version. `gestion 5.1` sin embargo es necesario corregir los datos que ya 
constan en el dataset. El Siguiente código identifica y corrige estas variaciones para tener valores unificados del tipo
de actividad

In [ ]:
"""
    El objetivo es identificar el tipo de actividad con un espacio al final 
    para ser reemplazadas por un mismo valor uniforme
    
    FILTROS:
    * Identificamos el texto: 'CORRECTIVAS '  "CORRECTIVO "
    
"""
mask = df['Tipo'] == "CORRECTIVO "
filtered_df = df[mask]
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo


In [ ]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Tipo'] = "CORRECTIVO"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 57 filas por modificar.
✅ Se ha actualizado el Dataframe


Lo mismo para las demas correcciones a realizar

In [ ]:
mask = df['Tipo'] == "ACTGIS"
filtered_df = df[mask]
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Tipo'] = "PREDICTIVO"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 0 filas por modificar.
✅ Se ha actualizado el Dataframe


### Identificar Cuentas de tipo "Servicios_Ocasionales"

In [ ]:
"""
    El objetivo es identificar el tipo de actividad (Cuenta) que se encuentran como desconocidas (?) 
    pero que se pueden atribuir como cuenta de tipo "Servicios_Ocasionales
    
    FILTROS:
    * Debe contar con un Alimentador identificado
    * Cuenta: Desconocida (?)
    * Tipo: RUTINARIA
    * Que en el texto (Evento) contengan las palabras, 'acometida' o 'medidor'

    Estas filas son de Tipo = Servicios_Ocasionales, 

    primero generamos una máscara para identificarlas, luego aplicamos el cambio de cuenta,
    verificamos y guardamos en el Dataset.
"""
cond1 = df['Alimentador'] != "·"
cond2 = df['Tipo'].str.contains("RUTINARIA", regex=False, na=False, case=False)
cond3 = df['Cuenta'] == "?" 
cond4 = df['Evento'].str.contains("s/o", regex=False, na=False, case=False)

# Combine the conditions with the "&" (AND) operator to create the final boolean mask
mask = cond1 & cond2 & cond3 & cond4

# Apply the mask to the DataFrame to get the filtered result
filtered_df = df[mask]

# Display the filtered DataFrame
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
25581,4,?,En Yantzaza en calle Jorge Mosquera en estruc...,NO PROG,Yantzaza III,No,No,No,RUTINARIA,·,...,2025-10-15 09:40:00,2025-10-15 10:10:00,30,BARRAZUETA GONZAGA SERVIO GUILLERMO,4,No,2-112,Yantzaza- Chicaña,163229,OT [04] Cuadrilla Yantzaza 2025-10-15 (021) SB...
26020,5,?,En el Barrio 2 de Noviembre se instala 4 S/O 2...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-10-31 11:50:00,2025-10-31 13:30:00,100,JARA NARVAEZ GALO SILVERIO,1,No,2-61,ZAMORA,164308,OT [21] Agencia Zamora 2025-10-31 (050) GJ.pdf
26073,7,?,En el Barrio Santa Elena se retira S/O en la C...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-11-07 13:15:00,2025-11-07 13:30:00,15,JARA NARVAEZ GALO SILVERIO,1,No,2-61,Zamora,164675,OT [21] Agencia Zamora 2025-11-07 (050) GJ.pdf
60691,10,?,En el Pangui se instala un S/O en el estructur...,PROG,El Pangui,No,No,No,RUTINARIA,·,...,2026-01-27 15:00:00,2026-01-27 15:20:00,20,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Uwents - Los Bayanes - San Roque,169627,OT [23] Agencia El Pangui 2026-01-27 (058) FA.pdf
153936,5,?,En Zamora parque Lineal se instala algunos S/O...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-11-07 11:25:00,2025-11-07 12:30:00,65,JARA NARVAEZ GALO SILVERIO,1,No,2-61,Zamora,164675,OT [21] Agencia Zamora 2025-11-07 (050) GJ.pdf
219573,11,?,En la Av Alonso de Mercadillo Parque Lineal se...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-11-07 15:30:00,2025-11-07 16:40:00,70,JARA NARVAEZ GALO SILVERIO,1,No,2-61,Zamora,164675,OT [21] Agencia Zamora 2025-11-07 (050) GJ.pdf
231724,3,?,En San Roque se instala un S/O en el estructur...,PROG,Los Encuentros,No,No,No,RUTINARIA,·,...,2025-10-16 08:50:00,2025-10-16 09:30:00,40,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pángui - Las Orquideas - Pachicutza - Pachkius,163310,OT [23] Agencia El Pangui 2025-10-16 (058) FA.pdf
264504,2,?,En el Centro Comercial se retira S/O 25 metros...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-10-31 09:30:00,2025-10-31 10:00:00,30,JARA NARVAEZ GALO SILVERIO,1,No,2-61,ZAMORA,164308,OT [21] Agencia Zamora 2025-10-31 (050) GJ.pdf
275979,10,?,Se instala un S/O en el estructura 72271,PROG,El Pangui,No,No,No,RUTINARIA,·,...,2025-11-14 15:40:00,2025-11-14 15:55:00,15,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - La Recta del Pangui - Guisme - Mia...,165125,OT [23] Agencia El Pangui 2025-11-14 (058) FA.pdf
297131,5,?,En Rancho Alegre de Cumbaratza se Notifica a l...,PROG,Zamora II,No,No,No,RUTINARIA,·,...,2025-11-18 12:00:00,2025-11-18 12:15:00,15,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, Timbara, Rancho Alegre, Descanso",165280,OT [21] Agencia Zamora 2025-11-18 (050) GJ.pdf


In [ ]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Cuenta'] = "Servicios_Ocasionales"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 33 filas por modificar.
✅ Se ha actualizado el Dataframe


### Identificar Cuentas de tipo "MEDIDORES"

In [ ]:
"""
    El objetivo es identificar el tipo de actividad (Cuenta) desconocidas (?) para separarlas de las que nos interesan (REDES)
    
    FILTROS:
    * Alimentador Zamora I
    * Tipo: Correctiva o Preventiva
    * Cuenta: Desconocida (?)
    * Que en el texto (Evento) contengan la palabra, 'medidor'

    Estas filas son de Tipo = MEDIDORES, 

    primero generamos una máscara para identificarlas, luego aplicamos el cambio de cuenta,
    verificamos y guardamos en el Dataset.
"""
mask = (
    (df['Alimentador'] != "·") &
    
    (
        df['Tipo'].str.contains("CORRECTIVO", regex=False, na=False, case=False) |
        df['Tipo'].str.contains("PREVENTIVO", regex=False, na=False, case=False)
    ) &
    
    (df['Cuenta'] == "?") &
    
    (
        df['Evento'].str.contains("medidor", regex=False, na=False, case=False)  # "acometida"
    )
)

# Apply the mask to the DataFrame to get the filtered view (optional)
filtered_df = df[mask]
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
15,5,?,Se llega la lugar y en la estructura #246959 s...,NO PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2026-02-23 11:40:00,2026-02-23 11:55:00,15,AMARI ORDONEZ JUNIOR IVAN,3,No,R-184,"Yantzaza, Zumbi, Paquisha, Nuevo Quito, Bella ...",171357,OT [06] Cuadrilla Paquisha 2026-02-23 (034) JA...
18,8,?,Se llega la lugar y en la estructura #32117 se...,NO PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2026-02-23 14:00:00,2026-02-23 16:00:00,120,AMARI ORDONEZ JUNIOR IVAN,3,No,R-184,"Yantzaza, Zumbi, Paquisha, Nuevo Quito, Bella ...",171357,OT [06] Cuadrilla Paquisha 2026-02-23 (034) JA...
22,14,?,Se llega al lugar y en la estructura #33697 se...,NO PROG,La Saquea,No,No,No,CORRECTIVO,·,...,2026-02-23 18:00:00,2026-02-23 18:10:00,10,AMARI ORDONEZ JUNIOR IVAN,3,Si,R-184,"Yantzaza, Zumbi, Paquisha, Nuevo Quito, Bella ...",171357,OT [06] Cuadrilla Paquisha 2026-02-23 (034) JA...
46,2,?,En el barrio La Pista se revisa variación de v...,NO PROG,Zamora II,No,No,No,PREVENTIVO,·,...,2026-02-23 09:30:00,2026-02-23 10:50:00,80,ORTEGA SERRANO STALIN JAVIER,1,No,2-61,"Zamora, San Marcos",171370,OT [21] Agencia Zamora 2026-02-23 (051) SO.pdf
119,14,?,"En Tundayme, estructura. 244700 se revisa el m...",NO PROG,Bomboiza,No,No,No,PREVENTIVO,·,...,2026-02-23 16:30:00,2026-02-23 16:40:00,10,MENDIETA MENDIETA HENRRY ALEXANDER,2,No,2-102,"Chuchumbletza, Gualaquiza, El Pangui.",171341,OT [08] Cuadrilla El Pangui 2026-02-23 (042) H...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410985,9,?,En el Sector La Chacra se reubica y se Restitu...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2026-02-11 15:15:00,2026-02-11 18:10:00,175,JARA NARVAEZ GALO SILVERIO,1,Si,2-61,"Zamora, Timbara, Cuzuntza",170699,OT [21] Agencia Zamora 2026-02-11 (050) GJ.pdf
410993,6,?,En San Ramón de Guadalupe se revisa 3 medidore...,PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2026-02-10 12:00:00,2026-02-10 12:30:00,30,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, San Antonio, Guadalupe, Kantzama, Guag...",170607,OT [21] Agencia Zamora 2026-02-10 (050) GJ.pdf
410996,9,?,En el Sector El Guayabal de Guadalupe se revis...,PROG,Yacuambi,No,No,No,PREVENTIVO,·,...,2026-02-10 13:30:00,2026-02-10 14:00:00,30,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, San Antonio, Guadalupe, Kantzama, Guag...",170607,OT [21] Agencia Zamora 2026-02-10 (050) GJ.pdf
411009,5,?,En la Quebrada de Cumbaratza se se Reubica y s...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2026-02-09 11:00:00,2026-02-09 13:00:00,120,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"ZAMORA, SAN MARCOS, QUEBRADA DE CUMBARATZA, CU...",170562,OT [21] Agencia Zamora 2026-02-09 (050) GJ.pdf


In [ ]:
# Actualizamos los valores de Cuenta en el dataframe completo para luego guardarlo en el Datalake

# df.loc[mask, 'Cuenta'] = "ACOMETIDAS"
filtered_df.loc[mask,'Cuenta'] = "MEDIDORES"
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
15,5,MEDIDORES,Se llega la lugar y en la estructura #246959 s...,NO PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2026-02-23 11:40:00,2026-02-23 11:55:00,15,AMARI ORDONEZ JUNIOR IVAN,3,No,R-184,"Yantzaza, Zumbi, Paquisha, Nuevo Quito, Bella ...",171357,OT [06] Cuadrilla Paquisha 2026-02-23 (034) JA...
18,8,MEDIDORES,Se llega la lugar y en la estructura #32117 se...,NO PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2026-02-23 14:00:00,2026-02-23 16:00:00,120,AMARI ORDONEZ JUNIOR IVAN,3,No,R-184,"Yantzaza, Zumbi, Paquisha, Nuevo Quito, Bella ...",171357,OT [06] Cuadrilla Paquisha 2026-02-23 (034) JA...
22,14,MEDIDORES,Se llega al lugar y en la estructura #33697 se...,NO PROG,La Saquea,No,No,No,CORRECTIVO,·,...,2026-02-23 18:00:00,2026-02-23 18:10:00,10,AMARI ORDONEZ JUNIOR IVAN,3,Si,R-184,"Yantzaza, Zumbi, Paquisha, Nuevo Quito, Bella ...",171357,OT [06] Cuadrilla Paquisha 2026-02-23 (034) JA...
46,2,MEDIDORES,En el barrio La Pista se revisa variación de v...,NO PROG,Zamora II,No,No,No,PREVENTIVO,·,...,2026-02-23 09:30:00,2026-02-23 10:50:00,80,ORTEGA SERRANO STALIN JAVIER,1,No,2-61,"Zamora, San Marcos",171370,OT [21] Agencia Zamora 2026-02-23 (051) SO.pdf
119,14,MEDIDORES,"En Tundayme, estructura. 244700 se revisa el m...",NO PROG,Bomboiza,No,No,No,PREVENTIVO,·,...,2026-02-23 16:30:00,2026-02-23 16:40:00,10,MENDIETA MENDIETA HENRRY ALEXANDER,2,No,2-102,"Chuchumbletza, Gualaquiza, El Pangui.",171341,OT [08] Cuadrilla El Pangui 2026-02-23 (042) H...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410985,9,MEDIDORES,En el Sector La Chacra se reubica y se Restitu...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2026-02-11 15:15:00,2026-02-11 18:10:00,175,JARA NARVAEZ GALO SILVERIO,1,Si,2-61,"Zamora, Timbara, Cuzuntza",170699,OT [21] Agencia Zamora 2026-02-11 (050) GJ.pdf
410993,6,MEDIDORES,En San Ramón de Guadalupe se revisa 3 medidore...,PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2026-02-10 12:00:00,2026-02-10 12:30:00,30,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, San Antonio, Guadalupe, Kantzama, Guag...",170607,OT [21] Agencia Zamora 2026-02-10 (050) GJ.pdf
410996,9,MEDIDORES,En el Sector El Guayabal de Guadalupe se revis...,PROG,Yacuambi,No,No,No,PREVENTIVO,·,...,2026-02-10 13:30:00,2026-02-10 14:00:00,30,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, San Antonio, Guadalupe, Kantzama, Guag...",170607,OT [21] Agencia Zamora 2026-02-10 (050) GJ.pdf
411009,5,MEDIDORES,En la Quebrada de Cumbaratza se se Reubica y s...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2026-02-09 11:00:00,2026-02-09 13:00:00,120,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"ZAMORA, SAN MARCOS, QUEBRADA DE CUMBARATZA, CU...",170562,OT [21] Agencia Zamora 2026-02-09 (050) GJ.pdf


### Guardar cambios en Deltalake

In [ ]:
import pyarrow as pa 

edited_data = pa.Table.from_pandas( filtered_df )

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
                    source=edited_data,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**")
except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")

✅ **Successfully saved changes to Delta Lake!**


## Mantenimiento `matriz_actividades`

### Limpieza de InicioEvento y FinEvento

In [ ]:
# 1. Elimina espacios en blanco del Inicio y del Fin del evento

df['InicioEvento'] = df['InicioEvento'].str.strip()
df['FinEvento'] = df['FinEvento'].str.strip()

In [ ]:
# Crear una mascara para los elementos que tienen 20 caracteres
# Create mask - True for rows that DON'T match the pattern
mask = ~df['FinEvento'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$')
# Handle NaN values explicitly
mask = mask | df['FinEvento'].isna()

In [ ]:
df[mask].tail(5)

In [ ]:
# Procesar estas fechas y horas para que todas tengan el mismo formato:

# 2. Funcion para eliminar el componente de Time Zone
#    éste es introducido cuando en actividades no se consigue una 'fecha_moda' 
#    y es necesario utilizar la 'fecha' de hoja_uno.   

def elimina_timezone( fecha ):
  try:
    
    # Se elimina el componente de Zona Horaria
    fecha_inicio = fecha.replace('T', ' ').split()
    fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
    return fecha_inicio
  
  except:
    print(f"[X] No fue posible convertir la cadena de caracteres:  {fecha}")
    return fecha

df.loc[mask, 'InicioEvento'] = df.loc[mask, 'InicioEvento'].apply( lambda x: elimina_timezone(x))
df.loc[mask, 'FinEvento'] = df.loc[mask, 'FinEvento'].apply( lambda x: elimina_timezone(x))


In [ ]:
def calcular_minutos_transcurridos( start_times, end_times ):
  """
  This function works because subtracting two pandas Series of datetimes
  is a vectorized operation.
  """
  try:
    # Ensure columns are in datetime format first
    start_times = pd.to_datetime(start_times)
    end_times = pd.to_datetime(end_times)

    time_difference = end_times - start_times
    # Return the difference in minutes
    return (time_difference.dt.total_seconds() / 60).astype(int)
  except Exception as e:
    print(f" EXCEPTION:\n{e}")

In [ ]:
#  ANALIZAR UNA OT filtrando por su indice

filtered_df = df.query( f"id_ot == 160106" ).sort_values(by='Item')
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
394358,1,informativa,En la agencia de la EERSSA se coordina los tra...,PROG,·,No,No,No,RUTINARIA,·,...,2025-08-29 07:30:00,2025-08-29 07:40:00,10,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394359,2,transporte,Traslado desde la agencia de la EERSSA Zamora ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-08-29 07:40:00,2025-08-29 07:50:00,10,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394360,3,REDES,En el sector de la Fragancia se realiza lo sig...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-08-29 07:50:00,2025-08-29 12:55:00,305,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394361,4,lunch,Lunch en La Fragancia.,ALIMEN,·,No,No,No,LUNCH,·,...,2025-08-29 12:55:00,2025-08-29 13:55:00,60,MORALES RIVERA LUIS ALBERTO,2,No,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394362,5,ALUMBRADO,Sector de la Fragancia se continua con la repa...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-08-29 13:55:00,2025-08-29 19:29:00,334,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394363,7,se_labora,SE LABORA: LM y LL de 07:30 a 12:55 y de 13:55...,LABORA,·,No,No,No,·,·,...,2025-08-29 00:00:01,2025-08-29 00:00:02,0,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394364,8,se_labora,SE LABORA: RY se encuentra con reposo médico.\...,LABORA,·,No,No,No,·,·,...,2025-08-29 00:00:01,2025-08-29 00:00:02,0,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf


In [ ]:
#filtered_df.loc[93810, 'InicioEvento'] = '2021-04-23 22:10:00'
filtered_df['Duracion'] = calcular_minutos_transcurridos(
    filtered_df['InicioEvento'],
    filtered_df['FinEvento']
)

#### Para corregir la Fecha final cuando se coloca "00:00:00" en lugar de "23:59:00" al finalizar el día

In [ ]:
# Mascara para determinar duracion menor a 0, para volver a calcular las horas. 

mask = ( df['Duracion'] < 0 )

In [ ]:
# Crear una mascara para aplicar los cambios

# Create mask for rows that end with '00:00:00'
mask = df['Duracion'].astype(str).str.endswith('00:00:00') & df['FinEvento'].notna() & ( df['Duracion'] < -1300 )

In [ ]:
# Muestra cuantos casos se ha identificado

true_indices = mask[mask].index
len(true_indices.tolist())

0

### Vuelve a calcular la columna 'Duracion' en minutos

In [ ]:
# Ejecuta el reemplazo de las horas

#df.loc[mask, 'FinEvento'] = df.loc[mask, 'FinEvento'].astype(str).str[:-8] + '23:59:00'
df['Duracion'] = calcular_minutos_transcurridos(
    df['InicioEvento'],
    df['FinEvento']
)

### Muestra fechas de actividades con posible conflicto

In [ ]:
# filtered_df = df.query("FinEvento.str.endswith('00:00:00') and Alimentador != '·' ")
# filtered_df = df.query("Duracion < 0")

filtered_df = df.query("Duracion < 0")
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
264193,12,transporte,Traslado desde el sector de La Quebrada de Cum...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-01-08 15:40:00,2026-01-08 11:56:00,-224,MORALES RIVERA LUIS ALBERTO,2,No,2-91,Zamora,168456,OT [02] Alumbrado Zamora 2026-01-08 (012) LM.pdf
276335,2,ALUMBRADO,"R.-Pangui,1lum150w,apagada,pst 127974 se cambi...",PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2026-01-22 19:45:00,2026-01-22 10:30:00,-555,ORELLANA BRAVO JORGE LUIS,3,No,2-30,EL PANGUI,169361,OT [no] Cuadrilla Loja 2026-01-22 (0) JOB.pdf
297095,3,lunch,Lunch en La Y del Guismi.,ALIMEN,·,No,No,No,LUNCH,·,...,2026-01-29 12:00:00,2026-01-29 00:00:00,-720,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"El Pangui, Gualaquiza",169771,OT [02] Alumbrado Zamora 2026-01-29 (012) LM.pdf
360569,22,ALUMBRADO,Zamora barrio Benjamín Carrión estructura. Nro...,PROG,Zamora II,No,No,No,CORRECTIVO,·,...,2026-01-28 16:52:00,2026-01-28 16:13:00,-39,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"Zamora, Timbara, Cumbaratza",169710,OT [02] Alumbrado Zamora 2026-01-28 (012) LM.pdf
364292,12,ALUMBRADO,Gualaquiza barrio Las Orquídeas estructura. 27...,PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-01-29 16:00:00,2026-01-29 15:19:00,-41,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"El Pangui, Gualaquiza",169771,OT [02] Alumbrado Zamora 2026-01-29 (012) LM.pdf
367707,3,?,"Portón estructura 128404 se revisa medidor, se...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-02-05 09:52:00,2026-02-05 08:52:00,-60,GUZMAN BARROS MARCO FERNANDO,1,No,2-111,"Gualaquiza, San José de Piunts, La Esperanza.",170310,OT [09] Cuadrilla Gualaquiza 2026-02-05 (045) ...
385217,5,ALUMBRADO,"INC No. 1103888976, Atendido, luminaria de 150...",PROG,La Saquea,No,No,No,CORRECTIVO,·,...,2026-02-02 16:25:00,2026-02-02 15:40:00,-45,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"Zamora, Yantzaza, Mercadillo",169907,OT [02] Alumbrado Zamora 2026-02-02 (012) LM.pdf
395669,18,transporte,Traslado a Gualaquiza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-02 21:45:00,2026-02-02 10:01:00,-704,GUZMAN BARROS MARCO FERNANDO,1,No,2-111,"Gualaquiza, San Francisco, Guayusal, Rosario, ...",170019,OT [09] Cuadrilla Gualaquiza 2026-02-02 (045) ...
399317,9,?,En Jembuentza se revisa medidor por no registr...,PROG,Yacuambi,No,No,No,PREVENTIVO,·,...,2026-01-30 13:30:00,2026-01-30 13:20:00,-10,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Yacuambi, Zamora.",169876,OT [21] Agencia Zamora 2026-01-30 (050) GJ.pdf
410827,5,transporte,"DAÑO REPORTADO POR CENTRO DE CONTROL, mediant...",TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-14 19:30:00,2026-02-14 10:10:00,-560,CHAMBA CANGO PEDRO ROSALINO,1,Si,2-112,"YANTZAZA, PINCHO.",170886,OT [22] Agencia Yantzaza 2026-02-14 (055) PCH.pdf


In [ ]:
df.query(f"Alimentador != '·'").sort_values(by="Duracion",ascending=True).head(10)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
94962,1,REDES,DEL CENTRO DE CONTROL INFORMAN QUE EN LOS SECT...,INFO,Y DE,No,No,No,·,·,...,2020-01-01 00:00:01,2020-01-01 00:00:02,0,LITUMA CORDOVA CESAR RUBEN,1,Si,R-12,GUALAQUIZA.GUAYUZAL,40977,OT [24] Agencia Gualaquiza 2020-01-01 (043) CL...
163532,1,REDES,"CALL CENTER LOJA INFORMA QUE EN CHUCHUMBLETZA,...",INFO,RAN,No,No,No,·,·,...,2018-11-10 00:00:01,2018-11-10 00:00:02,0,AMARI ORDONEZ JUNIOR IVAN,1,Si,R-96,"El Pangui, Chuchumbletza, El Pincho",19165,OT [08] Cuadrilla El Pangui 2018-11-10 (031) J...
163675,11,MEDIDORES,"En el Domicilio del Sr. Oclides Sosa, medidor ...",PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2021-10-27 14:20:00,2021-10-27 14:20:00,0,CHAMBA CANGO PEDRO ROSALINO,1,No,4-36,"Yantzaza, Panguintza",75209,OT [22] Agencia Yantzaza 2021-10-27 (052) PCH.pdf
389212,9,?,Del Centro de Control reportan S/S en el barr...,NO PROG,El Pangui,No,No,No,PREDICTIVO,·,...,2020-10-28 10:50:00,2020-10-28 10:50:00,0,CUENCA MURILLO JORGE VICENTE,2,No,,"El Pangui, El Guismi",55253,OT [23] Agencia El Pangui 2020-10-28 (0) JCM.pdf
339228,1,ALUMBRADO,Paquisha en la estructura. No. 29560 se arregl...,PROG,Paquisha,No,No,No,CORRECTIVO,·,...,2023-11-28 08:00:00,2023-11-28 08:00:00,0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"Paquisha, Santa Rosa, Nuevo Quito, Mayaycu, La...",119442,OT [02] Alumbrado Zamora 2023-11-28 (012) LM.pdf
94633,1,?,DE CALL CENTER LOJA INFORMAN QUE EN EL SECTOR...,INFO,P,No,No,No,·,·,...,2018-12-08 00:00:01,2018-12-08 00:00:02,0,LITUMA CORDOVA CESAR RUBEN,1,Si,R-101,LA MISIÓN DE BOMBOIZA,20536,OT [24] Agencia Gualaquiza 2018-12-08 (043) CL...
322635,1,?,"POR DISPOSICIÓN DE LA EERSSA, SE LABORA EN UNA...",INFO,Gualaquiza,No,No,No,·,·,...,2019-12-24 00:00:01,2019-12-24 00:00:02,0,RIOS RIOS FRANCISCO FERNANDO,0,Si,R-18,"Gualaquiza, Zamora",40645,OT [01] Cuadrilla Zamora 2019-12-24 (006) FR.pdf
360624,18,ALUMBRADO,INC. Nro. 1103745315. Gualaquiza barrio San Fr...,PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2025-05-27 16:55:00,2025-05-27 16:55:00,0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,Gualaquiza,153869,OT [02] Alumbrado Zamora 2025-05-27 (012) LM.pdf
76878,21,se_labora,SE LABORA: HM; VC de 20:04 a 21:07.,LABORA,El Pangui 2,No,No,No,·,·,...,2023-12-15 00:00:01,2023-12-15 00:00:02,0,MENDIETA MENDIETA HENRRY ALEXANDER,2,Si,2-102,"Chayazapa, Las Orquídeas, Shaime, Tsarunts.",120495,OT [08] Cuadrilla El Pangui 2023-12-15 (039) H...
294473,1,?,CALL CENTER LOJA INFORMA QUE EN EL BARRIO 8 DE...,INFO,A,No,No,No,·,·,...,2019-03-23 00:00:01,2019-03-23 00:00:02,0,AMARI ORDONEZ JUNIOR IVAN,1,Si,R-96,8 de Diciembre,25863,OT [08] Cuadrilla El Pangui 2019-03-23 (031) J...


In [ ]:
df

# LEGACY

> Analisis posterio entre MongoDB y DeltaLake

## Conectar con DASK Local Cluster

In [11]:
# DASK
from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()
dask.dashboard_link

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41201 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45905
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:41201/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39789'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43903'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45879'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39097'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:3979

'http://127.0.0.1:41201/status'

## Recargar Librerias Dinámicamente


In [6]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as ClaseOT
from eerssa import procesarOt as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import generarMatrizActividades as Actividades     # process ot.data["actividades"]
from eerssa import procesarActividades as ActividadesV30

In [13]:
reload( OrdenTrabajo )
reload( Actividades  )
reload( ClaseOT )
reload( ActividadesV30)

<module 'eerssa.procesarActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/procesarActividades.py'>

## Verificacion de OT desde MongoDB hacia DeltaLake

### Descargar y procesar [01] desde MongoDB 

In [ ]:
id_descarga = 97355

try:
  ot_test = CurrentCollection.find_one({'id_ot':id_descarga})
  if not ot_test:
    print(f" [X] No se pudo descargar la OT")
  else:
    activ = ActividadesV30.ConvertirOT_a_ActividadesCSV( ClaseOT.GestionOt.from_v30( ot_test ) )

except Exception as e:
  print(f"{e}")

In [35]:
import pyarrow as pa 

edited_data = pa.Table.from_pandas( activ )

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
                    source=edited_data,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .when_not_matched_by_source_delete(  # Rule 3: If an old activity is now gone...
                    predicate=f"target.id_ot = {id_descarga}"  # ...delete it, but only for the current OT.
                )
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**")
except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")

✅ **Successfully saved changes to Delta Lake!**


### Cursor para obtener todos los "id_ot" desde MongoDB

In [14]:
""" 
   OBTENER TODOS LOS 'id_ot' desde MongoDB
"""

try:
    # 1. Use a projection to only retrieve the 'id_ot' field.
    #    - {'id_ot': 1} means "include this field".
    #    - {'_id': 0} means "exclude the default _id field".
    cursor = CurrentCollection.find({}, {'id_ot': 1, '_id': 0})

    # 2. Create a list from the cursor results using a list comprehension.
    #    This iterates through each document in the cursor and extracts 'id_ot'.
    id_ot_list = [doc['id_ot'] for doc in cursor]

    # 3. Now you have your list of all 'id_ot' values.
    print(f"Successfully retrieved {len(id_ot_list)} 'id_ot' values.")
    if id_ot_list:
        print("First 10 values:", id_ot_list[:10])

except Exception as e:
    print(f"An error occurred: {e}")


An error occurred: name 'CurrentCollection' is not defined


In [15]:
"""
   Obtener todos los 'id_ot' existentes en DeltaLake
"""

delta_ids = df["id_ot"].unique()
len(delta_ids)

2203

In [ ]:
"""
   Difentecia de las ot que faltan en DeltaLake
"""
set_mongo = set(id_ot_list)
set_delta = set(delta_ids)

# Find which items in set_delta are not in set_mongo
new_ids_set = set_mongo.difference(set_delta)

# Convert the result back to a list
new_ids_to_process = list(new_ids_set)

print(f"Found {len(new_ids_to_process)} new IDs to be processed.")
# We sort the list here just for a predictable, clean output
print(f"New IDs: {sorted(new_ids_to_process)}")

In [ ]:
"""
   Descargar y procesar las OT faltantes y añadirlas al Delta Lake
"""
new_data_frames = []
for ot in new_ids_to_process:
  json_ot = CurrentCollection.find_one({"id_ot": ot})
  if not json_ot:
    print(f"No se pudo encontrar la OT con id_ot '{ot}' en MongoDB. Saltando.")
    continue
                
  obj_ot = OrdenTrabajo.GestionOt.from_dict(json_ot)
  new_data_frames.append(Actividades.ConvertirOT_a_ActividadesCSV(obj_ot))

In [ ]:
try:
  new_df = pd.concat(new_data_frames, ignore_index=True)
  write_deltalake(table_path, new_df, mode='append')
  print(f" [ EXITO ] DELTA LAKE Se han añadido {len(new_df)} filas a la tabla Delta en '{table_path}'.")
except Exception as e:
  print(f"Fallo al escribir en la tabla Delta: {e}")

 [ EXITO ] DELTA LAKE Se han añadido 121556 filas a la tabla Delta en './test/deltalake_2025'.


## FULL MONGO DOWNLOAD

Generar un nuevo archivo Delta Lake para unificar versiones - Ejecutado JULIO 2025

### Descarga de OT's desde MongoDB hacia Pickle y Delta Lake 

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

# ... setup client, db, collection
cursor = CurrentCollection.find()
all_documents = cursor.to_list() 
# or simply: all_documents = list(cursor)
print(f"Loaded {len(all_documents)} documents into a list.")
client.close()



Loaded 21879 documents into a list.


In [ ]:
obj_list = []
for document in all_documents:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot) )
df = pd.concat(obj_list, ignore_index=True)


In [ ]:
df["Fecha"] = pd.to_datetime(df["Fecha"])
df["InicioEvento"] = pd.to_datetime(df["InicioEvento"], format='mixed')
df["FinEvento"] = pd.to_datetime(df["FinEvento"], format='mixed')

df.to_pickle("/home/vlad/Documents/mongodb_v23.pkl")

/tmp/ipykernel_316235/3892611823.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["InicioEvento"] = pd.to_datetime(df["InicioEvento"], format='mixed')
/tmp/ipykernel_316235/3892611823.py:3: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["FinEvento"] = pd.to_datetime(df["FinEvento"], format='mixed')


In [ ]:
write_deltalake("/home/vlad/delta_v23", df)

In [ ]:
dt = DeltaTable("/home/vlad/delta_v23")

## Descargar OT faltantes desde MongoDB hacia DeltaLake

## Cargar Pickle para analisis

In [43]:
# DELTA LAKE Connection
# Verify the existence of the DELTA LAKE table
import pandas as pd
import numpy as np
from deltalake import DeltaTable
from datetime import datetime
from pprint import pprint


def borra_time_zone( fecha:str ):
  """
  Esta función elimina el componenete de Time Zone y deja solamente la fecha y hora. 
  En caso de que no contenga este componente deja el String intacto. 
  """
  fecha_inicio = fecha.replace('T', ' ').split()
  fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
  return fecha_inicio

if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")

df.info()

Conectado a la tabla Delta Lake en: /home/vlad/delta_V30
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31308 entries, 0 to 31307
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           31308 non-null  int64 
 1   Cuenta         31308 non-null  object
 2   Evento         31308 non-null  object
 3   Actividad      31308 non-null  object
 4   Alimentador    31308 non-null  object
 5   Primario       31308 non-null  object
 6   Desconexion    31308 non-null  object
 7   SIG            31308 non-null  object
 8   Tipo           31308 non-null  object
 9   Materiales     31308 non-null  object
 10  Cuadrilla      31308 non-null  object
 11  Dia            31308 non-null  object
 12  Fecha          31308 non-null  object
 13  InicioEvento   31308 non-null  object
 14  FinEvento      31308 non-null  object
 15  Duracion       31308 non-null  int64 
 16  Responsable    31308 non-null  object
 17  Colaboradore

In [31]:
variant = df.copy()

In [44]:
df['InicioEvento'] = df['InicioEvento'].apply( lambda x: borra_time_zone(x))
df['FinEvento'] = df['FinEvento'].apply( lambda x: borra_time_zone(x))

In [34]:
dt.version()

147

In [ ]:
# Perform the merge operation
print("\n--- Merging changes back into Delta Table ---")

(
    dt.merge(
        source=df,
        predicate="target.id = source.id",
        source_alias="source",
        target_alias="target"
    )
    .when_matched_update_all()  # If id matches, update the row
    .when_not_matched_insert_all()  # If a new id is in the source, insert it
    .execute()
)

print("Merge complete.")

### ¿Son todas los items en 'Fecha' validos?

In [45]:
def is_valid_utc_format(text_input):
    """
    Checks if a string can be converted to a timezone-aware datetime.

    The function tests against a specific ISO 8601 format that includes a
    UTC offset, like "2024-05-31T00:00:00-05:00".

    Args:
        text_input: The string or value to check.

    Returns:
        - True: if the input is a string and matches the format.
        - False: if the input is not a string or does not match the format.
        - pd.NaT: if the input is a null-like value (e.g., None, np.nan).
    """
    # 1. Handle null-like inputs first
    if pd.isna(text_input):
        return pd.NaT

    # 2. Ensure the input is a string before attempting to parse
    if not isinstance(text_input, str):
        return False

    # 3. Try to parse the string using the specific format
    try:
        # The format string matches the user's example.
        # %Y: 4-digit year
        # %m: 2-digit month
        # %d: 2-digit day
        # T: Literal 'T' separator
        # %H:%M:%S: Hour, minute, second
        # %z: UTC offset (e.g., -0500). Pandas extends this to handle
        #     the colon format (-05:00) as well.
        # errors='raise' ensures that any parsing failure raises an exception.
        pd.to_datetime(text_input, format="%Y-%m-%d %H:%M:%S", errors='raise')
        return True
    except ValueError:
        # This exception is raised if the string does not match the format.
        return False



In [47]:
df['InicioEvento'][0]

'2024-07-17 08:00:00'

In [37]:
fechas_validas = variant['InicioEvento'].apply( lambda x: is_valid_utc_format(x) )
fechas_validas.unique()

array([ True, False])

In [39]:
false_indices = np.where(~fechas_validas)[0]
len(false_indices)

18151

In [41]:
last_wrong_date = false_indices[-1]

In [ ]:
# 2023-02-20T00:00:00-05:00 00:00:01

In [42]:
variant.iloc[last_wrong_date]

Item                                                             4
Cuenta                                                   se_labora
Evento           SE LABORA: RM, RY\nNo se presentan novedades e...
Actividad                                                   LABORA
Alimentador                                                      ·
Primario                                                        No
Desconexion                                                     No
SIG                                                             No
Tipo                                                             ·
Materiales                                                       ·
Cuadrilla                                         Zamora (Agencia)
Dia                                                        viernes
Fecha                                    2022-02-11T00:00:00-05:00
InicioEvento                    2022-02-11T00:00:00-05:00 00:00:01
FinEvento                       2022-02-11T00:00:00-05:00 00:0

In [ ]:
import datetime
# 2. Define a function to safely get the date
def safe_to_date(value):
    # Check if the value is a Timestamp or datetime object
    if isinstance(value, (pd.Timestamp, datetime.datetime)):
        return value.date()
    # If it's already a date object, just return it
    elif isinstance(value, datetime.date):
        return value
    # For any other type, return NaT (Not a Time)
    else:
        return pd.NaT

In [ ]:
df['dates_equal_Inicio'] = (df['Fecha'].dt.date == df['InicioEvento'].apply(safe_to_date))
df['dates_equal_Fin'] = (df['Fecha'].dt.date == df['FinEvento'].apply(safe_to_date))

In [ ]:
df['dates_equal_Inicio'].unique()

array([ True, False])

In [ ]:
falla_inicio = df.query("dates_equal_Inicio == False")
#falla_inicio[["Item","Responsable","id_ot","Fecha","InicioEvento","FinEvento"]]
falla_inicio["id_ot"].unique()

array([155505, 155762, 145572, 144647, 148359, 147423, 148549, 148277,
       148246, 146584, 147455, 147804, 148333, 149748, 149726, 149242,
       150570, 137497, 139625, 138768, 142840, 139823, 139690, 134776,
       137787, 128220, 139615, 144309, 141353, 144417, 134989, 143002,
       140128, 134479, 137253, 125688, 110252, 105181, 114346, 119028,
       116773, 120598, 114096, 102758, 106048, 115879, 113463, 100339,
       104339, 111938, 120285, 114645, 109933, 111049,  84875,  89820,
        85621,  96068,  80898,  83812,  82240,  92859,  81700,  86257,
        83222, 156391,  75212,  67725,  76277,  73194,  70460,  76774,
        67486,  60815,  64571,  54263,  50513])

## VERIFICACIÓN de OTs en Mongo DB

1. Se extrae el listado de todos los `id_ot` de los PDF existentes
2. Se verifica este listado con los documentos en `MongoDB`
3. Se verifica este listado con los documentos en `DeltaLake`

### Conexion con DASK

In [18]:
# 1. Listado de OTs con sus ID:

# Directorio Raiz de las OT (año) para validar

#root_dir = ("/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/"
#            +
#            "2020")

root_dir = "/home/vlad/Documents/000 OTs Antiguas/2020"

list_pdfs = []
for path in Path( root_dir ).glob("**/*.pdf"):
    list_pdfs.append( str(path) )
    list_pdfs.sort()

print(f" Se han encontrado un total de: {len(list_pdfs)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# Helper function to call the method on the result of a future
def call_load_ot(orden_trabajo_object):
    """
    Takes the result of the first task (an OrdenTrabajo object) 
    and calls the load_ot() method on it.
    """
    return orden_trabajo_object.load_ot()

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.GestionOt, file) for file in list_pdfs]

# 2. Submit the second batch of tasks, feeding the first futures as input
futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step2)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")




 Se han encontrado un total de: 3054 Ordenes de Trabajo
Hora de inicio: 2025-07-24 10:21:50




   Procesados todos los 3054 items. Tiempo transcurrido: 294.74 segundos.
   Hora Final : 2025-07-24 10:26:45
